<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 3.2: 生成器: 集合
**上一节: [生成器: 参数](3.1_parameters.ipynb)**<br>
**下一节: [插曲: Chisel 标准库](3.2_interlude.ipynb)**


## 动机
生成器经常需要处理可变数量的对象，无论是 IOs、模块还是测试向量。
集合是处理这种情况的重要构建块。
本模块将介绍 Scala 集合以及如何将它们与 Chisel 生成器一起使用。

## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

注意我们在这里添加了一个新的导入，因为 `mutable.ArrayBuffer` 位于 `scala.collections` 中。

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test
import scala.collection._

---
# 生成器和集合<a name="generators-and-collections"></a> 
在本节中，我们将重点介绍*生成器*的概念以及使用 Scala 集合作为实现它们的工具。
我们不再将 Chisel 代码视为电路的*实例*，即特定电路的描述，
而是将其视为电路的生成器。

我们将从考虑之前练习中的 FIR 滤波器开始。

In [ ]:
class My4ElementFir(b0: Int, b1: Int, b2: Int, b3: Int) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(8.W))
    val out = Output(UInt(8.W))
  })

  val x_n1 = RegNext(io.in, 0.U)
  val x_n2 = RegNext(x_n1, 0.U)
  val x_n3 = RegNext(x_n2, 0.U)
  io.out := io.in * b0.U(8.W) + x_n1 * b1.U(8.W) +
    x_n2 * b2.U(8.W) + x_n3 * b3.U(8.W)
}


这个电路是生成器的一个简单案例，因为它可以生成具有不同系数的 4 抽头滤波器版本。
但是如果我们希望电路有更多抽头怎么办？我们将分几个步骤来完成这个任务。

- 构建一个抽头可配置 FIR 的软件*黄金模型*。
- 重新设计我们的测试以使用这个模型，并确认它有效。
- 重构我们的 My4ElementFir 以允许可配置的抽头数量。
- 使用我们新的测试框架测试新电路。

<span style="color:blue">**示例: FIR 黄金模型**</span><br><a name="fir-golden-model"></a> 
下面是一个 FIR 电路的 Scala 软件实现。

In [ ]:
/**
  * A naive implementation of an FIR filter with an arbitrary number of taps.
  */
class ScalaFirFilter(taps: Seq[Int]) {
  var pseudoRegisters = List.fill(taps.length)(0)

  def poke(value: Int): Int = {
    pseudoRegisters = value :: pseudoRegisters.take(taps.length - 1)
    var accumulator = 0
    for(i <- taps.indices) {
      accumulator += taps(i) * pseudoRegisters(i)
    }
    accumulator
  }
}

### Seq
请注意 `taps` has become a `Seq[Int]` which means that the user of the 类 can pass an arbitrarily-long sequence of `Int`s when constructing the 类.
### Registers
With `  var pseudoRegisters = List.fill(taps.length)(0)` we create a `List` that will hold values from previous cycles.  `List` was chosen because its syntax of adding an element to the head and removing the last element is very simple. Just about any member of the scala collections family could be used.  We are also initializing this list to contain all zeros.
### 注入
Our 类 adds a 注入 函数/方法 that emulates putting a new 输入 into the filter and cycling the 时钟.
### Updating the registers
The line `pseudoRegisters = 值 :: pseudoRegisters.take(taps.length - 1)` first uses the `take` 方法 of list to keep the all but the last element of the list, then uses the `::` list concatentation operator to add `值` to the head of the reduced version of the list.
### Computing the 输出
A simple for loop with an accumulator sums each element of the list times its corresponding tap coefficient. The line with just `accumulator` returns that 值 as the 函数 result.
## Adapting our previous 测试 for testing our golden model
We will now use our previous work to confirm that our golden model works.  A bit of editing magic takes our previous tests harness and morphs it into...

In [ ]:
val filter = new ScalaFirFilter(Seq(1, 1, 1, 1))

var out = 0

out = filter.poke(1)
println(s"out = $out")
assert(out == 1)  // 1, 0, 0, 0

out = filter.poke(4)
assert(out == 5)  // 4, 1, 0, 0
println(s"out = $out")

out = filter.poke(3)
assert(out == 8)  // 3, 4, 1, 0
println(s"out = $out")

out = filter.poke(2)
assert(out == 10)  // 2, 3, 4, 1
println(s"out = $out")

out = filter.poke(7)
assert(out == 16)  // 7, 2, 3, 4
println(s"out = $out")

out = filter.poke(0)
assert(out == 12)  // 0, 7, 2, 3
println(s"out = $out")

Executing the previous block demonstrates that our 软件 model returns the same results as My4ElementFir did.


## 测试 电路 using the golden model.<a name="use-golden-model-as-测试"></a> 
Now that we are reasonably confident about our golden model, we re-write our 测试 to compare the 电路 outputs with the 输出 of the golden model, instead of using laboriously hand-crafted 示例.
What follows is a quick first pass to do it.

In [ ]:
val goldenModel = new ScalaFirFilter(Seq(1, 1, 1, 1))

test(new My4ElementFir(1, 1, 1, 1)) { c =>
    for(i <- 0 until 100) {
        val input = scala.util.Random.nextInt(8)

        val goldenModelResult = goldenModel.poke(input)

        c.io.in.poke(input.U)

        c.io.out.expect(goldenModelResult.U, s"i $i, input $input, gm $goldenModelResult, ${c.io.out.peek().litValue}")

        c.clock.step(1)
    }

}


Our 测试 runs for 100 cycles, and checks that the two different methods, 硬件 and 软件, are in sync at each 步进.

### Things to watch out for
(i.e., mistakes we actually committed while writing this.)

1. Getting the 步进 in the right place. 软件 and 硬件 execute differently; it's easy to get this wrong.
1. This 测试 is weak because it is very sensitive to how the IOs and registers are sized. Implementing a 软件 golden model that observes wrapping behavior at arbitrary data bit widths can be complicated.  Here we just make sure that we only pass in values that fit.

<span style="color:blue">**示例: Parameterized FIR Generator**</span><br><a name="fir-golden-model"></a> 
Below we have created a new Filter 类, `MyManyElementsFilter` that takes a `Seq` of constants to use for taps.  This list can be any number of elements.
For good measure a `bitWidth` has been added that allows us to control the sizes of numbers that can be handled by our 电路.
In response the 变量 length we have had to refactor the creation of registers and how they are connected.
The methodology used below uses a simple subset of the available library of collection functions.
Later sections show how to more succinctly express the behavior in a way that also makes what is happening clearer.

In [ ]:
class MyManyElementFir(consts: Seq[Int], bitWidth: Int) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(bitWidth.W))
    val out = Output(UInt(bitWidth.W))
  })

  val regs = mutable.ArrayBuffer[UInt]()
  for(i <- 0 until consts.length) {
      if(i == 0) regs += io.in
      else       regs += RegNext(regs(i - 1), 0.U)
  }
  
  val muls = mutable.ArrayBuffer[UInt]()
  for(i <- 0 until consts.length) {
      muls += regs(i) * consts(i).U
  }

  val scan = mutable.ArrayBuffer[UInt]()
  for(i <- 0 until consts.length) {
      if(i == 0) scan += muls(i)
      else scan += muls(i) + scan(i - 1)
  }

  io.out := scan.last
}

#### How we did it
There are three parallel sections starting at lines 7, 13, and 18.
We are using a Scala collection 类型 called `ArrayBuffer`.
`ArrayBuffer` allows you to append elements using the `+=` operator (also insert and delete, but we don't need this).
First, we create an ArrayBuffer `regs` whose elements will be `UInt`s.
Then iterate over the taps, adding the 输入 as the first element followed by creating registers using RegNext which connect the 输入 of the 寄存器 to the previous element (`regs(i-1)`) and initializes it to unsigned zero (`0.U`).
These registers will hold the previous values of inputs as they are needed.

Next, we create another ArrayBuffer `muls` of `UInt`s.
Each element of muls will be a node whose i-th element is the product of the `regs(i)` and `const(i)`.

Note the use of the `scan.last` 方法.
It takes the last element of a collection, and is a more elegant alternative to `regs(i - 1)` used during the `regs` construction.

### Does it behave the same as `My4ElementFir`?
A good first 测试 of our new version is to see if it can pass the 测试 we just applied to the
`My4ElementFir`.
We create an 实例 of `MyManyElementFir` and run even more data through it.

In [ ]:
val goldenModel = new ScalaFirFilter(Seq(1, 1, 1, 1))

test(new MyManyElementFir(Seq(1, 1, 1, 1), 8)) { c =>
    for(i <- 0 until 100) {
      val input = scala.util.Random.nextInt(8)

      val goldenModelResult = goldenModel.poke(input)

      c.io.in.poke(input.U)

      c.io.out.expect(goldenModelResult.U, s"i $i, input $input, gm $goldenModelResult, ${c.io.out.peek().litValue}")

      c.clock.step(1)
    }
}

### Now let's 测试 a bunch of different sized FIR filters
We create some helper functions: `r` which gets a random number; `runOneTest` which creates a golden model and a 硬件 仿真 of a filter for a particular set of taps, and then runs at least twice the number of taps worth of data through the filter.

In [ ]:
/** a convenience method to get a random integer
  */
def r(): Int = {
  scala.util.Random.nextInt(1024)
}

/**
  * run a test comparing software and hardware filters
  * run for at least twice as many samples as taps
  */
def runOneTest(taps: Seq[Int]) {
    val goldenModel = new ScalaFirFilter(taps)

    test(new MyManyElementFir(taps, 32)) { c =>
        for(i <- 0 until 2 * taps.length) {
            val input = r()

            val goldenModelResult = goldenModel.poke(input)

            c.io.in.poke(input.U)

            c.io.out.expect(goldenModelResult.U, s"i $i, input $input, gm $goldenModelResult, ${c.io.out.peek().litValue}")

            c.clock.step(1)
        }
    }
}

for(tapSize <- 2 until 100 by 10) {
    val taps = Seq.fill(tapSize)(r())  // create a sequence of random coefficients

    runOneTest(taps)
}

### Just for fun, let's make a bigger one
以下 will run a single 测试 on a 500 tap
FIR filter.  It can take a minute or so to run.
(Hint: Watch for the Scala ● to change to Scala ○ on the Toolbar when the execution completes.)

In [ ]:
runOneTest(Seq.fill(500)(r()))

---
# 硬件 Collections

<span style="color:blue">**示例: Add run-time configurable taps to our FIR**</span><br>
以下 code adds an additional `consts` vector to the IO of our FIR generator which allows the coefficients to be changed externally after 电路 generation.
This is done with the Chisel collection 类型 `Vec`.
`Vec` supports many of the scala collection methods but it can only contain Chisel 硬件 elements.
`Vec` should only be used in situations where ordinary Scala collections won't work.  
基本上 this is in one of two situations.
1. You need a collection of elements in a 束, typically a 束 that will be used as IO.
1. You need to access the collection via an index 也就是说 part of the 硬件 (think 寄存器 File).


In [ ]:
class MyManyDynamicElementVecFir(length: Int) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(8.W))
    val out = Output(UInt(8.W))
    val consts = Input(Vec(length, UInt(8.W)))
  })

  // Reference solution
  val regs = RegInit(VecInit(Seq.fill(length - 1)(0.U(8.W))))
  for(i <- 0 until length - 1) {
      if(i == 0) regs(i) := io.in
      else       regs(i) := regs(i - 1)
  }
  
  val muls = Wire(Vec(length, UInt(8.W)))
  for(i <- 0 until length) {
      if(i == 0) muls(i) := io.in * io.consts(i)
      else       muls(i) := regs(i - 1) * io.consts(i)
  }

  val scan = Wire(Vec(length, UInt(8.W)))
  for(i <- 0 until length) {
      if(i == 0) scan(i) := muls(i)
      else scan(i) := muls(i) + scan(i - 1)
  }

  io.out := scan(length - 1)
}

In [ ]:
val goldenModel = new ScalaFirFilter(Seq(1, 1, 1, 1))

test(new MyManyDynamicElementVecFir(4)) { c =>
    c.io.consts(0).poke(1.U)
    c.io.consts(1).poke(1.U)
    c.io.consts(2).poke(1.U)
    c.io.consts(3).poke(1.U)
    for(i <- 0 until 100) {
        val input = scala.util.Random.nextInt(8)

        val goldenModelResult = goldenModel.poke(input)

        c.io.in.poke(input.U)

        c.io.out.expect(goldenModelResult.U, s"i $i, input $input, gm $goldenModelResult, ${c.io.out.peek().litValue}")

        c.clock.step(1)
    }
}


<span style="color:red">**练习: 32-bit RISC-V Processor**</span><br><a name="fir-golden-model"></a>

A [寄存器 file](https://en.wikipedia.org/wiki/Register_file) is an important building block for making a processor.
A 寄存器 file is an array of registers that can be read from or written to via a number of read or write ports.
Each port consists of an address and data field.

The [RISC-V instruction set architecture](https://riscv.org/specifications/) defines several variants, the simplest of which is called RV32I.
RV32I has a size-32 array of 32-bit registers.
**The 寄存器 at index 0 (the first 寄存器) is always zero when you read from it, regardless of what you write to it** (it's often useful to have 0 handy).

Implement a 寄存器 file for RV32I with a single write port and a paramterized number of read ports.
Writes will only be performed when `wen` (write enable) is asserted.

In [ ]:
class RegisterFile(readPorts: Int) extends Module {
    require(readPorts >= 0)
    val io = IO(new Bundle {
        val wen   = Input(Bool())
        val waddr = Input(UInt(5.W))
        val wdata = Input(UInt(32.W))
        val raddr = Input(Vec(readPorts, UInt(5.W)))
        val rdata = Output(Vec(readPorts, UInt(32.W)))
    })
    
    // A Register of a vector of UInts
    val reg = RegInit(VecInit(Seq.fill(32)(0.U(32.W))))
    
    ???

    
}

In [ ]:
test(new RegisterFile(2) ) { c =>
  def readExpect(addr: Int, value: Int, port: Int = 0): Unit = {
    c.io.raddr(port).poke(addr.U)
    c.io.rdata(port).expect(value.U)
  }
  def write(addr: Int, value: Int): Unit = {
    c.io.wen.poke(true.B)
    c.io.wdata.poke(value.U)
    c.io.waddr.poke(addr.U)
    c.clock.step(1)
    c.io.wen.poke(false.B)
  }
  // everything should be 0 on init
  for (i <- 0 until 32) {
    readExpect(i, 0, port = 0)
    readExpect(i, 0, port = 1)
  }

  // write 5 * addr + 3
  for (i <- 0 until 32) {
    write(i, 5 * i + 3)
  }

  // check that the writes worked
  for (i <- 0 until 32) {
    readExpect(i, if (i == 0) 0 else 5 * i + 3, port = i % 2)
  }
}

<div id="container"><section id="accordion"><div>
<输入 类型="checkbox" id="check-1" />
<label for="check-1"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
    when (io.wen) {
        reg(io.waddr) := io.wdata
    }
    for (i &lt;- 0 until readPorts) {
        when (io.raddr(i) === 0.U) {
            io.rdata(i) := 0.U
        } .otherwise {
            io.rdata(i) := reg(io.raddr(i))
        }
    }

</pre></article></div></section></div>

---
# You're done!

[Return to the top.](#top)